## Task 1.1: Train ANN on given dataset

Run given scrpit to create training data:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report

# im using google colab so need to do this
nltk.download('punkt_tab')
nltk.download('stopwords')



def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','')                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return data

# If this is the primary file that is executed (ie not an import of another file)
if __name__ == "__main__":
    # get data, pre-process and split
    data = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
    data.columns = ['Sentence', 'Class']
    data['index'] = data.index                                          # add new column index
    columns = ['index', 'Class', 'Sentence']
    data = preprocess_pandas(data, columns)                             # pre-process
    training_data, validation_data, training_labels, validation_labels = train_test_split( # split the data into training, validation, and test splits
        data['Sentence'].values.astype('U'),
        data['Class'].values.astype('int32'),
        test_size=0.10,
        random_state=0,
        shuffle=True
    )

    # vectorize data using TFIDF and transform for PyTorch for scalability
    word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
    training_data = word_vectorizer.fit_transform(training_data)        # transform texts to sparse matrix
    training_data = training_data.todense()                             # convert to dense matrix for Pytorch
    vocab_size = len(word_vectorizer.vocabulary_)
    validation_data = word_vectorizer.transform(validation_data)
    validation_data = validation_data.todense()
    train_x_tensor = torch.from_numpy(np.array(training_data)).type(torch.FloatTensor)
    train_y_tensor = torch.from_numpy(np.array(training_labels)).long()
    validation_x_tensor = torch.from_numpy(np.array(validation_data)).type(torch.FloatTensor)
    validation_y_tensor = torch.from_numpy(np.array(validation_labels)).long()


<>:26: SyntaxWarning: invalid escape sequence '\.'
<>:27: SyntaxWarning: invalid escape sequence '\w'
<>:28: SyntaxWarning: invalid escape sequence '\d'
<>:26: SyntaxWarning: invalid escape sequence '\.'
<>:27: SyntaxWarning: invalid escape sequence '\w'
<>:28: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_7430/1554519936.py:26: SyntaxWarning: invalid escape sequence '\.'
  data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
/tmp/ipykernel_7430/1554519936.py:27: SyntaxWarning: invalid escape sequence '\w'
  data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','')                                                       # remove special characters
/tmp/ipykernel_7430/1554519936.py:28: SyntaxWarning: invalid escape sequence '\d'
  data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
[nltk_data] Downloading packa

### Create ANN:

In [ ]:
# create ANN for text processing, and train it on the given data.

import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleANN(nn.Module):
  def __init__(self, vocab_size):
    super(SimpleANN,self).__init__()

    # input layer
    self.in_layer = nn.Linear(vocab_size,512)

    self.layer2 = nn.Linear(512,128)

    self.layer3 = nn.Linear(128,32)

    # output layer
    self.end_layer = nn.Linear(32,2)

  def forward(self,x):
    # do the correlation
    x = F.relu(self.in_layer(x))
    x = F.relu(self.layer2(x))
    x = F.relu(self.layer3(x))

    x = self.end_layer(x)

    return x


Train model on given data:

In [ ]:
import torch.optim as optim

# init model
model = SimpleANN(vocab_size)

# define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# training loop
epochs = 20
batch_size = 32

for epoch in range(epochs):
  model.train()

  outputs= model(train_x_tensor)
  loss = criterion(outputs, train_y_tensor)
  optimizer.zero_grad()
  loss.backward()

  optimizer.step()

  if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

RuntimeError: mat1 and mat2 must have the same dtype, but got Long and Float

Evaluate model:

In [ ]:
model.eval()
with torch.no_grad():
    val_outputs = model(validation_x_tensor)
    _, predicted = torch.max(val_outputs, 1)

    correct = (predicted == validation_y_tensor).sum().item()
    accuracy = correct / validation_y_tensor.size(0)
    tp = ((predicted == 1) & (validation_y_tensor == 1)).sum().item()
    fp = ((predicted == 1) & (validation_y_tensor == 0)).sum().item()
    fn = ((predicted == 0) & (validation_y_tensor == 1)).sum().item()

    # Add a small epsilon to avoid division by zero errors if the model predicts all 0s
    epsilon = 1e-7
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)

    f1 = 2 * (precision * recall) / (precision + recall + epsilon)

    print(f"Acc: {accuracy:.2f}% | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


Acc: 0.72% | Precision: 1.0000 | Recall: 0.4717 | F1: 0.6410


1K training: Around 80-85 accuracy and simular F1 score

### Create RNN with LSTM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleLSTM(nn.Module):
    def __init__(self, vocab_size):
        super(SimpleLSTM, self).__init__()

        # input layer
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=512)
       # self.in_layer = nn.Linear(vocab_size, 512)

        # lstm layer1
        self.lstm1 = nn.LSTM(input_size=512, hidden_size=128, batch_first=True)

        # lstm layer2
        self.lstm2 = nn.LSTM(input_size=128, hidden_size=32, batch_first=True)

        # output layer
        self.end_layer = nn.Linear(32, 2)

    def forward(self, x):

        # Apply the linear transformation to the sequence
        #x = F.relu(self.in_layer(x))
        x = self.embedding(x)

        # Pass through the First LSTM
        x, _ = self.lstm1(x)

        # Pass through the Second LSTM
        lstm_out, (h_n, c_n) = self.lstm2(x)

        # slice tensor
        x = lstm_out[:,-1, :]

        # Pass through the final output layer
        x = self.end_layer(x)

        # Final shape: (batch_size, 2)
        return x

Train it on data

In [ ]:
import torch.optim as optim

# init model

model_LSTM = SimpleLSTM(vocab_size=vocab_size)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_LSTM.to(device)

# Encode and Pad Sequences
MAX_SEQ_LENGTH = 50 # Adjust based on your average sentence length
X_train_pad = encode_and_pad(X_train_text, vocab_to_int, MAX_SEQ_LENGTH)
X_val_pad = encode_and_pad(X_val_text, vocab_to_int, MAX_SEQ_LENGTH)

# define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_LSTM.parameters(), lr=0.001)

# Convert to PyTorch Tensors as longs
train_x_tensor = torch.from_numpy(X_train_pad).long()
train_y_tensor = torch.from_numpy(y_train).long()
val_x_tensor = torch.from_numpy(X_val_pad).long()
val_y_tensor = torch.from_numpy(y_val).long()

# Create DataLoaders
batch_size = 64
train_dataset = TensorDataset(train_x_tensor, train_y_tensor)
val_dataset = TensorDataset(val_x_tensor, val_y_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

for epoch in range(epochs):
  # --- TRAINING ---
  model_LSTM.train()
  total_loss = 0
  for inputs, targets in train_loader:
    inputs,targets = inputs.to(device), targets.to(device)
    outputs = model_LSTM(inputs.long())
    loss = criterion(outputs, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
      print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [5/20], Loss: 0.6951
Epoch [5/20], Loss: 0.6896
Epoch [5/20], Loss: 0.6969
Epoch [5/20], Loss: 0.6937
Epoch [5/20], Loss: 0.6944
Epoch [5/20], Loss: 0.6932
Epoch [5/20], Loss: 0.6935
Epoch [5/20], Loss: 0.6921
Epoch [5/20], Loss: 0.6937
Epoch [5/20], Loss: 0.6905
Epoch [10/20], Loss: 0.6943
Epoch [10/20], Loss: 0.6929
Epoch [10/20], Loss: 0.6934
Epoch [10/20], Loss: 0.6922
Epoch [10/20], Loss: 0.6926
Epoch [10/20], Loss: 0.6932
Epoch [10/20], Loss: 0.6941
Epoch [10/20], Loss: 0.6914
Epoch [10/20], Loss: 0.6912
Epoch [10/20], Loss: 0.6934
Epoch [15/20], Loss: 0.6887
Epoch [15/20], Loss: 0.6939
Epoch [15/20], Loss: 0.6973
Epoch [15/20], Loss: 0.6932
Epoch [15/20], Loss: 0.6930
Epoch [15/20], Loss: 0.6928
Epoch [15/20], Loss: 0.6931
Epoch [15/20], Loss: 0.6932
Epoch [15/20], Loss: 0.6934
Epoch [15/20], Loss: 0.6933
Epoch [20/20], Loss: 0.6924
Epoch [20/20], Loss: 0.6946
Epoch [20/20], Loss: 0.6916
Epoch [20/20], Loss: 0.6894
Epoch [20/20], Loss: 0.6944
Epoch [20/20], Loss: 0.6983
Ep

Eval the model:

In [ ]:
model_LSTM.eval()
with torch.no_grad():
    val_input = validation_x_tensor.unsqueeze(1)
    val_outputs = model_LSTM(val_input)
    _, predicted = torch.max(val_outputs, 1)

    correct = (predicted == validation_y_tensor).sum().item()
    accuracy = correct / validation_y_tensor.size(0)
    tp = ((predicted == 1) & (validation_y_tensor == 1)).sum().item()
    fp = ((predicted == 1) & (validation_y_tensor == 0)).sum().item()
    fn = ((predicted == 0) & (validation_y_tensor == 1)).sum().item()

    # Add a small epsilon to avoid division by zero errors if the model predicts all 0s
    epsilon = 1e-7
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)

    f1 = 2 * (precision * recall) / (precision + recall + epsilon)

    print(f"Acc: {accuracy:.2f}% | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


NameError: name 'validation_x_tensor' is not defined

1K training: Around 80% accuracy and F1


# Train on more data
Create new data set


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import nltk
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords
from nltk import word_tokenize
from collections import Counter
from torch.utils.data import TensorDataset, DataLoader

nltk.download('punkt_tab')
nltk.download('stopwords')

def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace('[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)                      # remove emails
    data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
    data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','')                                                       # remove special characters
    data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return data


def encode_and_pad(sentences, vocab_to_int, max_seq_length):
    """Converts sentences to integer sequences and pads them to max_seq_length."""
    features = np.zeros((len(sentences), max_seq_length), dtype=int)
    for i, sentence in enumerate(sentences):
        # Convert words to ints; if word not in vocab, ignore or map to 0
        int_seq = [vocab_to_int.get(word, 0) for word in sentence.split()]
        # Truncate if longer than max_seq_length
        int_seq = int_seq[:max_seq_length]
        # Insert into the features matrix (pads with 0s automatically)
        features[i, :len(int_seq)] = int_seq
    return features


# Load Data
data = pd.read_csv("amazon_cells_labelled_LARGE_25K.txt", delimiter='\t', header=None)
data.columns = ['Sentence', 'Class']
data['index'] = data.index                                          # add new column index
columns = ['index', 'Class', 'Sentence']

# Pre-process text
data = preprocess_pandas(data, columns)

# Split Data
X_train_text, X_val_text, y_train, y_val = train_test_split(
    data['Sentence'].values.astype('U'),
    data['Class'].values.astype('int32'),
    test_size=0.60,
    random_state=0,
    shuffle=True
  )

print(len(X_train_text))

# Build Vocabulary from Training Data
all_words = ' '.join(X_train_text).split()
word_counts = Counter(all_words)
# Sort words by frequency
sorted_vocab = sorted(word_counts, key=word_counts.get, reverse=True)

# Integer mapping
vocab_to_int = {word: idx + 1 for idx, word in enumerate(sorted_vocab)}

# Size of vocab needs to account for padding
vocab_size = len(vocab_to_int) + 1

# Encode and Pad Sequences
MAX_SEQ_LENGTH = 25
X_train_pad = encode_and_pad(X_train_text, vocab_to_int, MAX_SEQ_LENGTH)
X_val_pad = encode_and_pad(X_val_text, vocab_to_int, MAX_SEQ_LENGTH)

# Convert to PyTorch Tensors as longs
train_x_tensor = torch.from_numpy(X_train_pad).long()
train_y_tensor = torch.from_numpy(y_train).long()
val_x_tensor = torch.from_numpy(X_val_pad).long()
val_y_tensor = torch.from_numpy(y_val).long()

# Create DataLoaders
batch_size = 64
train_dataset = TensorDataset(train_x_tensor, train_y_tensor)
val_dataset = TensorDataset(val_x_tensor, val_y_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)



<>:19: SyntaxWarning: invalid escape sequence '\.'
<>:20: SyntaxWarning: invalid escape sequence '\w'
<>:21: SyntaxWarning: invalid escape sequence '\d'
<>:19: SyntaxWarning: invalid escape sequence '\.'
<>:20: SyntaxWarning: invalid escape sequence '\w'
<>:21: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_743/725644970.py:19: SyntaxWarning: invalid escape sequence '\.'
  data['Sentence'] = data['Sentence'].replace('((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)    # remove IP address
/tmp/ipykernel_743/725644970.py:20: SyntaxWarning: invalid escape sequence '\w'
  data['Sentence'] = data['Sentence'].str.replace('[^\w\s]','')                                                       # remove special characters
/tmp/ipykernel_743/725644970.py:21: SyntaxWarning: invalid escape sequence '\d'
  data['Sentence'] = data['Sentence'].replace('\d', '', regex=True)                                                   # remove numbers
[nltk_data] Downloading package pun

10000


Train on new data.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EmbeddedLSTM(nn.Module):
    def __init__(self, vocab_size):
        super(EmbeddedLSTM, self).__init__()

        # Input Layer: Embedding
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=512)

        # lstm layer1
        self.lstm1 = nn.LSTM(input_size=512, hidden_size=32, batch_first=True)

        # lstm layer2
        self.lstm2 = nn.LSTM(input_size=128, hidden_size=32, batch_first=True)

        # output layer
        self.end_layer = nn.Linear(32, 2)

    def forward(self, x):
        # Pass through embedding layer
        x = self.embedding(x)

        # Pass through the First LSTM
      #  x, _ = self.lstm1(x)

        # Pass through the Second LSTM
        lstm_out, (h_n, c_n) = self.lstm1(x)

        # slice tensor
        x = lstm_out[:,-1, :]

        # Pass through the final output layer
        x = self.end_layer(x)

        # Final shape: (batch_size, 2)
        return x

class EmbeddedANN(nn.Module):
    def __init__(self, vocab_size):
        super(EmbeddedANN, self).__init__()

        # Input Layer: Embedding
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=512)

        # Hidden Layers
        self.layer2 = nn.Linear(512, 128)
        self.layer3 = nn.Linear(128, 32)

        # Output Layer
        self.end_layer = nn.Linear(32, 2)

    def forward(self, x):
        # x expected shape: (batch_size, sequence_length) containing integer IDs.

        # Convert integers to dense vectors
        x = self.embedding(x)

        # squich
        x = x.mean(dim=1)

        # Pass through the rest of original ANN
        x = F.relu(self.layer2(x))
        x = F.relu(self.layer3(x))
        x = self.end_layer(x)

        # Final shape: (batch_size, 2)
        return x



Train embedded models on large data

In [ ]:
def train_model(model, train_loader, epochs=5, lr=0.001):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    print(f"--- Training {model.__class__.__name__} ---")

    for epoch in range(epochs):
        # --- TRAINING ---
        model.train()
        total_loss = 0
        for inputs, targets in train_loader:
            inputs,targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    print("-" * 40 + "\n")
    return model

# Run the training
trained_lstm = train_model(EmbeddedLSTM(vocab_size), train_loader)
trained_ann = train_model(EmbeddedANN(vocab_size), train_loader)
#trained_lstm_1K =train_model(SimpleLSTM(vocab_size), train_loader)


NameError: name 'vocab_size' is not defined

eval models

In [ ]:
 # --- VALIDATION ---
def eval_model(model, val_loader):
  model.eval()
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  tp, fp, fn, tn = 0, 0, 0, 0

  with torch.no_grad():
    for inputs, targets in val_loader:

      targets = targets.to(device)

      if isinstance(inputs, dict):
        # for transformer
        inputs = {k: v.to(device) for k, v in inputs.items()}
        outputs = model(**inputs)
      else:
        # for regular NNs
        inputs = inputs.to(device)
        outputs = model(inputs)

      if isinstance(outputs, torch.Tensor):
        logits = outputs                 # Standard PyTorch models
      elif hasattr(outputs, "logits"):
        logits = outputs.logits          # Hugging Face object
      elif isinstance(outputs, tuple):
        logits = outputs[0]              # Hugging Face tuple
      else:
        logits = outputs

      _, predicted = torch.max(logits, 1)



      # Logic for TP, FP, FN, TN
      tp += ((predicted == 1) & (targets == 1)).sum().item()
      fp += ((predicted == 1) & (targets == 0)).sum().item()
      fn += ((predicted == 0) & (targets == 1)).sum().item()
      tn += ((predicted == 0) & (targets == 0)).sum().item()

  # Calculate Precision, Recall, and F1
  # Added 1e-7 to denominators to prevent "Division by Zero" errors
  precision = tp / (tp + fp + 1e-7)
  recall = tp / (tp + fn + 1e-7)
  f1 = 2 * (precision * recall) / (precision + recall + 1e-7)

  accuracy = (tp + tn) / (tp + tn + fp + fn)
  print(f"Acc: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

print("LSTM")
eval_model(trained_lstm, val_loader)
#eval_model(trained_lstm_1K, val_loader)
print("ANN")
eval_model(trained_ann, val_loader)

LSTM
Acc: 0.8381 | Precision: 0.8548 | Recall: 0.8854 | F1: 0.8698
ANN
Acc: 0.8390 | Precision: 0.8705 | Recall: 0.8652 | F1: 0.8679


## Task 1.2: Implement Transformer

Following https://pytorch.org/hub/huggingface_pytorch-transformers/ to implement

In [ ]:
# install requrements
!pip install tqdm boto3 requests regex sentencepiece sacremoses
!pip install pytorch-transformers


In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
import torch

tokenizer = AutoTokenizer.from_pretrained('bert-base-cased-finetuned-mrpc')

val_encodings = tokenizer(
    X_val_text.tolist(),
    #truncation=True,
    padding=True,
    max_length=24,       # Using your chosen MAX_SEQ_LENGTH
    return_tensors='pt'  # Return PyTorch tensors
)

class TransformerDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Package input_ids and attention_mask into a dictionary
        inputs = {key: val[idx] for key, val in self.encodings.items()}
        target = self.labels[idx]
        return inputs, target

    def __len__(self):
        return len(self.labels)

# 4. Create the Dataset and DataLoader
# We use y_val directly from your train_test_split
val_dataset_transformer = TransformerDataset(val_encodings, y_val)
val_loader_transformer = DataLoader(val_dataset_transformer, batch_size=64, shuffle=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2402: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


In [ ]:
from transformers import AutoModelForSequenceClassification

# Load the model
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Pass your NEW dataloader into the evaluation function
eval_model(model, val_loader_transformer)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Acc: 0.6110 | Precision: 0.6110 | Recall: 1.0000 | F1: 0.7585


# Task 1.3: Comparison
When we compare the models we see that for the small dataset it was alot eaiser to create a simple ANN that had okay results, around 80% on every metric(recall got close to 100% sometimes), while the LSTM needed more layers to not be underfitted for the these small NNs I'm creating.

For the large dataset embedding was required for both the ANN, and the LSTM. The ANN had about the same results as when trained on the small dataset. While the LSTM had a much better results when trained on the larger dataset even when underfitted.
This tells me that the ANN performs very well on smaller datasets and does not scale as well with dataset size, while LSTM needs a bit larger datasets to be trained correctly. After fixing the LSTM to have a better fit ---


The bert transformer had very high recall while lacking in precision, and accuracy.